# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [235]:
!cat publications.tsv

pub_date	title	venue	excerpt	abstract	citation	doi	url	url_slug	paper_url
2015-04-26	Performance analysis for overflow loss systems of processor-sharing queues	IEEE Conference on Computer Communications (INFOCOM)	Applies the Information Exchange Surrogate Approximation to the blocking probability evaluation of overflow loss networks of processor-sharing queues.	Overflow loss systems have wide applications in telecommunications and multimedia systems. In this paper, we consider an overflow loss system consisting of a set of finite-buffer processor-sharing (PS) queues, and develop effective methods for evaluation of its blocking probability. For such a problem, an existing approximation of the blocking probability is based on decomposition of the system into independent PS queues. We provide a new approximation which instead performs decomposition on a surrogate model of the original system, and demonstrate via extensive numerical results that our new approximation is more accurate and r

## Import pandas

We are using the very handy pandas library for dataframes.

In [236]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [237]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0, encoding='utf-8')
publications


,pub_date,title,venue,excerpt,abstract,citation,doi,url,url_slug,paper_url
0,2015-04-26,Performance analysis for overflow loss systems...,IEEE Conference on Computer Communications (IN...,Applies the Information Exchange Surrogate App...,Overflow loss systems have wide applications i...,"**Y.-C. Chan**, J. Guo, E. W. M. Wong, and M. ...",10.1109/INFOCOM.2015.7218518,NaN,chan2015_infocom,/files/chan2015_infocom.pdf
1,2016-07-07,Surrogate models for performance evaluation of...,Performance Evaluation,Applies the Information Exchange Surrogate App...,We consider a model of overflow loss systems i...,"**Y.-C. Chan**, J. Guo, E. W. M. Wong, and M. ...",10.1016/j.peva.2016.06.007,NaN,chan2016_peva,/files/chan2016_peva.pdf
2,2017-12-04,Energy efficiency-QoS tradeoff in cellular net...,GLOBECOM 2017 - 2017 IEEE Global Communication...,"Models a network of base stations, with each B...",Energy efficiency and Quality of Service (QoS)...,"J. Wu, E. W. M. Wong, **Y.-C. Chan**, and M. Z...",10.1109/GLOCOM.2017.8254042,NaN,wu2017_globecom,/files/wu2017_globecom.pdf
3,2017-12-18,Blocking probability evaluation for non-hierar...,IEEE Transactions on Communications,An in-depth introduction to the Information Ex...,Non-hierarchical overflow loss systems (NH-OLS...,"**Y.-C. Chan** and E. W. M. Wong, “Blocking pr...",10.1109/TCOMM.2017.2784450,NaN,chan2018_tcom,/files/chan2018_tcom.pdf
4,2017-07-28,Overflow models for the admission of intensive...,Health Care Management Science,Proposes a new model for a network of ICUs and...,"An earlier article, inspired by overflow model...","**Y.-C. Chan**, E. W. M. Wong, G. Joynt, P. La...",10.1007/s10729-017-9412-8,NaN,chan2017_hcms,/files/chan2017_hcms.pdf
5,2020-05-11,Performance Evaluation of 5G mmWave Networks w...,2020 IEEE 21st International Conference on Hig...,Proposes a model of a 5G millimetre-wave netwo...,We propose a versatile cross-layer framework t...,"J. Wu, M. Wang, **Y.-C. Chan**, E. W. M. Wong,...",10.1109/HPSR48589.2020.9098993,NaN,wu2020_hpsr,/files/wu2020_hpsr.pdf
6,2020-06-05,Power Consumption and GoS Tradeoff in Cellular...,IEEE Transactions on Green Communications and ...,"Models a network of base stations, with each B...",Mobile network operators usually consider powe...,"J. Wu, E. W. M. Wong, **Y.-C. Chan**, and M. Z...",10.1109/TGCN.2020.3000277,NaN,wu2020_tgcn,/files/wu2020_tgcn.pdf
7,2020-10-20,Modeling the COVID-19 Pandemic Using an SEIHR ...,IEEE Access,Proposes an SEIHR compartmental model with mig...,The 2019 novel coronavirus disease (COVID-19) ...,"R. Niu, E. W. M. Wong, **Y.-C. Chan**, M. A. V...",10.1109/ACCESS.2020.3032584,NaN,niu2020_access,/files/niu2020_access.pdf
8,2021-01-18,Evaluating Non-Hierarchical Overflow Loss Syst...,IEEE Communications Letters,Proposes hybrid learning combining the Informa...,The Information Exchange Surrogate Approximati...,"**Y.-C. Chan**, E. W. M. Wong, and C. S. Leung...",10.1109/LCOMM.2021.3052683,NaN,chan2021_comml,/files/chan2021_comml.pdf
9,2021-07-06,A stochastic SEIHR model for COVID-19 data flu...,Nonlinear Dynamics,Proposes a stochastic SEIHR model for COVID-19...,Although deterministic compartmental models ar...,"R. Niu, **Y.-C. Chan**, E. W. M. Wong, M. A. v...",10.1007/s11071-021-06631-9,NaN,niu2021_nody,/files/niu2021_nody.pdf


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [238]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [239]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.abstract)) > 5:
        md += "\n**Abstract:** " + html_escape(item.abstract) + "\n"
    
    if len(str(item.doi)) > 5:
        md += "\n**DOI**: [" + item.doi + "](https://doi.org/" + item.doi + ")\n"
    elif len(str(item.url)) > 5:
        md += "\n**URL**: <" + item.url + ">\n" 
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\n**Recommended citation**:<br/>" + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w', encoding='utf-8') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [240]:
!ls ../_publications/

2015-04-26-chan2015_infocom.md	2021-01-18-chan2021_comml.md
2016-07-07-chan2016_peva.md	2021-07-06-niu2021_nody.md
2017-07-28-chan2017_hcms.md	2022-07-06-wang2022_viruses.md
2017-12-04-wu2017_globecom.md	2023-06-07-wong2023_access.md
2017-12-18-chan2018_tcom.md	2023-07-10-mukherjee2023_ec3.md
2020-05-11-wu2020_hpsr.md	2023-09-25-moretti2023_lodisa.md
2020-06-05-wu2020_tgcn.md	2023-12-13-chan2023_wsc.md
2020-10-20-niu2020_access.md


In [241]:
!cat ../_publications/2015-04-26-chan2015_infocom.md

---
title: "Performance analysis for overflow loss systems of processor-sharing queues"
collection: publications
permalink: /publication/2015-04-26-chan2015_infocom
excerpt: 'Applies the Information Exchange Surrogate Approximation to the blocking probability evaluation of overflow loss networks of processor-sharing queues.'
date: 2015-04-26
venue: 'IEEE Conference on Computer Communications (INFOCOM)'
paperurl: '/files/chan2015_infocom.pdf'
citation: '**Y.-C. Chan**, J. Guo, E. W. M. Wong, and M. Zukerman, “Performance analysis for overflow loss systems of processor-sharing queues,” in *2015 IEEE Conference on Computer Communications (INFOCOM)*, (Hong Kong), IEEE, Apr. 2015.'
---
Applies the Information Exchange Surrogate Approximation to the blocking probability evaluation of overflow loss networks of processor-sharing queues.

**Abstract:** Overflow loss systems have wide applications in telecommunications and multimedia systems. In this paper, we consider an overflow loss system co